Trying different types of promts


In [ ]:
import os
from dotenv import load_dotenv
from pathlib import Path
from src.ingestion.pdf_parser import parse_pdf
from src.ingestion.chunker import load_and_chunk_pdf
from src.ingestion.embedder import embed_texts
from src.ingestion.ingest_pipeline import run_ingestion
from src.retrieval.retriever import retrieve
from src.retrieval.reranker import rerank
from src.retrieval.vector_store import VectorStore
from src.evaluation.evaluator import evaluate_rag

In [ ]:
load_dotenv()
ROOT = Path(os.getenv("PYTHONPATH"))
# Resolve path from project root (notebook often runs with cwd = notebooks/)
ROOT = Path.cwd()
if not (ROOT / "uploaded_pdfs").exists():
    ROOT = ROOT.parent

V1 - Promt(inbuilt at this stage)

In [ ]:
results = evaluate_rag(
    json_path=str(ROOT / "eval_data" / "test_questions.json"),
)

In [ ]:
print(results["scores"])

At first look they are: {'faithfulness': 0.9016, 'answer_relevancy': 0.9356, 'context_recall': 0.9058, 'context_precision': 0.7996}


V2- Promt

In [ ]:
system_prompt="You are a research assistant. Your ONLY source of information is the chunks provided below. Do not use any knowledge from your training, even if you know the answer. If the provided chunks do not contain enough information to answer, say exactly: 'The provided context does not contain this information.' Do not fabricate, infer, or extrapolate beyond what is explicitly stated in the chunks."


In [ ]:
results = evaluate_rag(
    json_path=str(ROOT / "eval_data" / "test_questions.json"),system_prompt= system_prompt,
)

In [ ]:
print(results["scores"])

The score for V2 promt:{'faithfulness': 0.9058, 'answer_relevancy': 0.8041, 'context_recall': 0.9275, 'context_precision': 0.7865} 

Stricter grounding improves faithfulness but hurts answer relevancy. The model stops hallucinating but also becomes too conservative — it sometimes refuses to answer when it could have.
There's a tradeoff. You can't maximize all 4 metrics with just a prompt change. Faithfulness and answer_relevancy pull in opposite directions. That's why LangGraph query rewriting exists in Phase 4 — it improves answer_relevancy without sacrificing faithfulness.


By claude ("Need to work on this a lot")

V3 - Promt

In [ ]:
system_prompt="You are a research assistant. Your ONLY source of information is the chunks provided below. Do not use any knowledge from your training, even if you know the answer. If the provided chunks do not contain enough information to answer, say exactly: 'The provided context does not contain this information.' Do not fabricate, infer, or extrapolate beyond what is explicitly stated in the chunks. When you use information in your answer, indicate which chunk it came from. Example: 'According to Chunk 2, the model achieves 94% accuracy.' If a sentence cannot be traced to a specific chunk, do not include it."

In [ ]:
results = evaluate_rag(
    json_path=str(ROOT / "eval_data" / "test_questions.json"),system_prompt= system_prompt,
)

In [ ]:
print(results["scores"])

For V3 promt - {'faithfulness': 0.8986, 'answer_relevancy': 0.8493, 'context_recall': 0.8841, 'context_precision': 0.7590}

Adding citation instructions actually hurt everything slightly. The model spent more effort formatting citations than answering well.
Winner is v2 — best faithfulness and context_recall. The answer_relevancy drop is acceptable and fixable in Phase 4 with query rewriting.

" By claude again"